In [1]:
from datetime import datetime
import json
from src.helpers import get_project_folders, normalize_project
from src.metric import TimestampMetric
from pathlib import Path
from tqdm.contrib.concurrent import process_map

def iterate_parallel(func, files: list[Path]) -> list:
    items = []
    for file in files:
        items.append((file.parent.name, file))
    results = process_map(func, items, chunksize=1)
    results = [result for result in results if result is not None]
    return [item for sublist in results if isinstance(sublist, list) for item in sublist] if results and isinstance(results[0], list) else results

/home/jortvd/Documents/GitKraken/rust-migration-thesis/analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INPUT_FOLDER = "/home/jortvd/thesis-data/github-results-25-3-2026"
OUTPUT_FOLDER = "../results"

In [3]:
def is_non_distribution_asset(asset: dict) -> bool:
    return asset["size"] < 128_000 or "source" in asset["name"]

In [4]:
metric = TimestampMetric("mean_distribution_size")

release_files = sorted([f for f in Path(INPUT_FOLDER).glob("**/release_*.json") if f.name.endswith(".json")])
def process_item(item):
    project_folder, release_file = item

    with release_file.open() as f:
        release = json.load(f)

    if sizes := [
        asset["size"]
        for asset in release.get("assets", [])
        if not is_non_distribution_asset(asset)
    ]:
        return (
            normalize_project(project_folder),
            datetime.fromisoformat(release["published_at"]),
            sum(sizes) / len(sizes),
            False
        )
    return None

metric.add_many(iterate_parallel(process_item, release_files))
metric.save(OUTPUT_FOLDER)

100%|██████████| 2486/2486 [00:00<00:00, 7288.70it/s]


In [5]:
PLATFORM_MAPPING = {
    "WINDOWS": ["windows", "win", "win32", "win64", "exe", "msi", "nsis", "mingw", "cygwin", "msix"],
    "DARWIN": ["darwin", "macos", "osx", "mac", "apple", "dmg", "app", "pkg"],
    "IOS": ["ios", "ipa", "iphoneos", "tvos", "watchos"],
    "LINUX": [
        "linux", "debian", "ubuntu", "centos", "fedora", "rhel", "alpine", "suse", 
        "deb", "rpm", "appimage", "snap", "flatpak", "musl", "arch"
    ],
    "BSD": ["freebsd", "openbsd", "netbsd", "dragonfly"],
    "ANDROID": ["android", "apk", "aab"],
    "WASM": ["wasm", "wasi", "webassembly"],
    "CROSS_PLATFORM": ["jar", "any", "js"],
    "SOURCE": ["src", "source"]
}

import re

def asset_name_to_platforms(name: str) -> list[str]:
    platforms = []
    tokens = set(re.split(r'[-_.\s+]', name.lower()))

    for platform, keywords in PLATFORM_MAPPING.items():
        if any(keyword in tokens for keyword in keywords) and platform != "SOURCE":
            platforms.append(platform)

    if len(platforms) > 1:
        print(f"Warning: asset '{name}' matches multiple platforms: {platforms}")

    return platforms

In [6]:
metric = TimestampMetric("distribution_platform_count")

def process_item(item):
    project_folder, release_file = item
    with release_file.open() as f:
        release = json.load(f)

    assets = release.get("assets", [])
    if len(assets) == 0:
        return None

    platforms = set()
    for asset in assets:
        if not is_non_distribution_asset(asset):
            platforms.update(asset_name_to_platforms(asset["name"]))

    return (
        normalize_project(project_folder),
        datetime.fromisoformat(release["published_at"]),
        len(platforms),
        False
    )

metric.add_many(iterate_parallel(process_item, release_files))
metric.save(OUTPUT_FOLDER)

100%|██████████| 2486/2486 [00:00<00:00, 7837.92it/s]
